# Filtered State Summaries

This notebook loads the filtered state paths from the paired Markovian SABR and Rough-SABR runs and creates summary tables and diagnostic figures.


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    return here.parent if here.name == "notebooks" else here


def project_relative(path: Path) -> str:
    path = Path(path).resolve()
    try:
        return str(path.relative_to(PROJECT_ROOT)).replace(chr(92), "/")
    except ValueError:
        return str(path)


PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
COMPARISON_ROOT = OUTPUT_ROOT / "comparisons" / "filtered_state_summaries"
TABLE_DIR = COMPARISON_ROOT / "tables"
FIGURE_DIR = COMPARISON_ROOT / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

NORMAL_PREFIX = "normal_sabr_400ts_no_A_RW_seed"
ROUGH_PREFIX = "rough_sabr_400ts_logU0_seed"
SEEDS = [123, 456, 789, 101112, 131415]
PRIMARY_SEED = 123


## Load State Paths

Load the filtered state paths by model and seed, then add the observed forward proxy for comparison.


In [ ]:
def load_state_paths(model_prefix: str, model_name: str) -> pd.DataFrame:
    parts = []
    for seed in SEEDS:
        path = OUTPUT_ROOT / f"{model_prefix}{seed}" / "filtered_state_path.csv"
        if not path.exists():
            print("Missing:", project_relative(path))
            continue
        h = pd.read_csv(path)
        h.insert(0, "seed", seed)
        h.insert(1, "model", model_name)
        parts.append(h)
    return pd.concat(parts, ignore_index=True)


def load_forward_proxy() -> pd.DataFrame:
    parts = []
    for seed in SEEDS:
        path = OUTPUT_ROOT / f"{NORMAL_PREFIX}{seed}" / "selected_observations.csv"
        if not path.exists():
            continue
        h = pd.read_csv(path, usecols=["t_index", "F_obs"])
        h = h.groupby("t_index", as_index=False)["F_obs"].median()
        h.insert(0, "seed", seed)
        parts.append(h)
    return pd.concat(parts, ignore_index=True)

normal_states = load_state_paths(NORMAL_PREFIX, "Markovian")
rough_states = load_state_paths(ROUGH_PREFIX, "Rough-SABR")
forward_proxy = load_forward_proxy()

state_paths = pd.concat([normal_states, rough_states], ignore_index=True, sort=False)
state_paths = state_paths.merge(forward_proxy, on=["seed", "t_index"], how="left")
state_paths["F_error_vs_obs"] = state_paths["F"] - state_paths["F_obs"]
state_paths["abs_F_error_vs_obs"] = state_paths["F_error_vs_obs"].abs()
state_paths["vol_level_proxy"] = state_paths["atm_logvol_proxy"]
state_paths["memory_norm_for_plot"] = state_paths.get("memory_norm", np.nan)

state_paths.to_csv(TABLE_DIR / "filtered_state_paths_paired.csv", index=False)
display(state_paths.head())


## State Summary Table

Summarize the main filtered state variables by model.


In [ ]:
summary_rows = []
for model, h in state_paths.groupby("model"):
    row = {
        "model": model,
        "n_seed_timestamp_obs": len(h),
        "mean_abs_F_error": h["abs_F_error_vs_obs"].mean(),
        "median_abs_F_error": h["abs_F_error_vs_obs"].median(),
        "mean_atm_logvol_proxy": h["atm_logvol_proxy"].mean(),
        "std_atm_logvol_proxy": h["atm_logvol_proxy"].std(),
        "mean_rho": h["rho"].mean(),
        "std_rho": h["rho"].std(),
        "mean_nu": h["nu"].mean(),
        "std_nu": h["nu"].std(),
        "mean_beta": h["beta"].mean(),
        "std_beta": h["beta"].std(),
        "mean_fit_rmse": h["fit_price_rmse"].mean(),
    }
    if "memory_norm" in h:
        row["mean_memory_norm"] = h["memory_norm"].mean()
        row["std_memory_norm"] = h["memory_norm"].std()
    summary_rows.append(row)

state_summary = pd.DataFrame(summary_rows)
state_summary.to_csv(TABLE_DIR / "filtered_state_summary_by_model.csv", index=False)
display(state_summary.round(6))


## Seed-Averaged State Paths

Plot the filtered forward, ATM log-volatility proxy, rho, and nu after averaging across seeds.


In [ ]:
timestamp_summary = (
    state_paths.groupby(["model", "t_index"], as_index=False)
    .agg(
        F_mean=("F", "mean"),
        F_std=("F", "std"),
        F_obs_mean=("F_obs", "mean"),
        atm_logvol_proxy_mean=("atm_logvol_proxy", "mean"),
        atm_logvol_proxy_std=("atm_logvol_proxy", "std"),
        rho_mean=("rho", "mean"),
        rho_std=("rho", "std"),
        nu_mean=("nu", "mean"),
        nu_std=("nu", "std"),
        beta_mean=("beta", "mean"),
        beta_std=("beta", "std"),
        fit_price_rmse_mean=("fit_price_rmse", "mean"),
    )
)
timestamp_summary.to_csv(TABLE_DIR / "filtered_state_paths_across_seed_summary.csv", index=False)

normal_ts = timestamp_summary[timestamp_summary["model"] == "Markovian"].sort_values("t_index")
rough_ts = timestamp_summary[timestamp_summary["model"] == "Rough-SABR"].sort_values("t_index")
forward_ts = state_paths.groupby("t_index", as_index=False)["F_obs"].mean()

normal_color = "#1c5982"
rough_color = "#d65225"
obs_color = "0.30"

fig, axes = plt.subplots(4, 1, figsize=(12.5, 9.0), sharex=True)

axes[0].plot(forward_ts["t_index"], forward_ts["F_obs"], color=obs_color, linewidth=1.15, label="Observed forward")
axes[0].plot(normal_ts["t_index"], normal_ts["F_mean"], color=normal_color, linewidth=1.15, label="Markovian")
axes[0].plot(rough_ts["t_index"], rough_ts["F_mean"], color=rough_color, linewidth=1.15, label="Rough-SABR")
axes[0].set_ylabel("Forward", fontsize=15)
axes[0].legend(loc="upper right", frameon=False, fontsize=11)

axes[1].plot(normal_ts["t_index"], normal_ts["atm_logvol_proxy_mean"], color=normal_color, linewidth=1.15, label="Markovian")
axes[1].plot(rough_ts["t_index"], rough_ts["atm_logvol_proxy_mean"], color=rough_color, linewidth=1.15, label="Rough-SABR")
axes[1].set_ylabel("ATM log-vol", fontsize=15)
axes[1].legend(loc="upper right", frameon=False, fontsize=11)

axes[2].plot(normal_ts["t_index"], normal_ts["rho_mean"], color=normal_color, linewidth=1.15, label="Markovian")
axes[2].plot(rough_ts["t_index"], rough_ts["rho_mean"], color=rough_color, linewidth=1.15, label="Rough-SABR")
axes[2].set_ylabel(r"$\rho$", fontsize=15)
axes[2].legend(loc="upper right", frameon=False, fontsize=11)

axes[3].plot(normal_ts["t_index"], normal_ts["nu_mean"], color=normal_color, linewidth=1.15, label="Markovian")
axes[3].plot(rough_ts["t_index"], rough_ts["nu_mean"], color=rough_color, linewidth=1.15, label="Rough-SABR")
axes[3].set_ylabel(r"$\nu$", fontsize=15)
axes[3].set_xlabel("Timestamp index", fontsize=17)
axes[3].legend(loc="upper right", frameon=False, fontsize=11)

for ax in axes:
    ax.grid(alpha=0.20)
    ax.tick_params(axis="both", which="major", direction="in", length=8, width=1.2, labelsize=13)
    ax.tick_params(axis="both", which="minor", direction="in", length=4, width=1.0)
    ax.minorticks_on()

fig.tight_layout()
path = FIGURE_DIR / "filtered_state_interpretable_paths_across_seeds.png"
fig.savefig(path, bbox_inches="tight", dpi=220)
plt.show()
print("Saved:", project_relative(path))


## Primary-Seed State Paths

Plot one paired seed and include the rough-memory norm.


In [ ]:
primary = state_paths[state_paths["seed"] == PRIMARY_SEED].sort_values(["model", "t_index"])
normal_primary = primary[primary["model"] == "Markovian"].sort_values("t_index")
rough_primary = primary[primary["model"] == "Rough-SABR"].sort_values("t_index")
forward_primary = primary.groupby("t_index", as_index=False)["F_obs"].mean()

fig, axes = plt.subplots(5, 1, figsize=(12.5, 10.2), sharex=True)

axes[0].plot(forward_primary["t_index"], forward_primary["F_obs"], color=obs_color, linewidth=1.15, label="Observed forward")
axes[0].plot(normal_primary["t_index"], normal_primary["F"], color=normal_color, linewidth=1.0, label="Markovian")
axes[0].plot(rough_primary["t_index"], rough_primary["F"], color=rough_color, linewidth=1.0, label="Rough-SABR")
axes[0].set_ylabel("Forward", fontsize=15)
axes[0].legend(loc="upper right", frameon=False, fontsize=11)

axes[1].plot(normal_primary["t_index"], normal_primary["atm_logvol_proxy"], color=normal_color, linewidth=1.0, label="Markovian")
axes[1].plot(rough_primary["t_index"], rough_primary["atm_logvol_proxy"], color=rough_color, linewidth=1.0, label="Rough-SABR")
axes[1].set_ylabel("ATM log-vol", fontsize=15)
axes[1].legend(loc="upper right", frameon=False, fontsize=11)

axes[2].plot(normal_primary["t_index"], normal_primary["rho"], color=normal_color, linewidth=1.0, label="Markovian")
axes[2].plot(rough_primary["t_index"], rough_primary["rho"], color=rough_color, linewidth=1.0, label="Rough-SABR")
axes[2].set_ylabel(r"$\rho$", fontsize=15)
axes[2].legend(loc="upper right", frameon=False, fontsize=11)

axes[3].plot(normal_primary["t_index"], normal_primary["nu"], color=normal_color, linewidth=1.0, label="Markovian")
axes[3].plot(rough_primary["t_index"], rough_primary["nu"], color=rough_color, linewidth=1.0, label="Rough-SABR")
axes[3].set_ylabel(r"$\nu$", fontsize=15)
axes[3].legend(loc="upper right", frameon=False, fontsize=11)

if "memory_norm" in rough_primary:
    axes[4].plot(rough_primary["t_index"], rough_primary["memory_norm"], color=rough_color, linewidth=1.0, label="Rough memory norm")
axes[4].set_ylabel("Memory", fontsize=15)
axes[4].set_xlabel("Timestamp index", fontsize=17)
axes[4].legend(loc="upper right", frameon=False, fontsize=11)

for ax in axes:
    ax.grid(alpha=0.20)
    ax.tick_params(axis="both", which="major", direction="in", length=8, width=1.2, labelsize=13)
    ax.tick_params(axis="both", which="minor", direction="in", length=4, width=1.0)
    ax.minorticks_on()

fig.tight_layout()
path = FIGURE_DIR / f"filtered_state_primary_seed_{PRIMARY_SEED}_with_memory.png"
fig.savefig(path, bbox_inches="tight", dpi=220)
plt.show()
print("Saved:", project_relative(path))


## Rolling-Window State Paths

Plot the filtered states for one rolling-window run.


In [ ]:
ROLLING_WINDOW_LABEL = "w301_700"
ROLLING_SEED = 123

rolling_root = OUTPUT_ROOT / "rolling_windows" / ROLLING_WINDOW_LABEL
normal_roll_dir = rolling_root / f"normal_sabr_no_A_RW_seed{ROLLING_SEED}"
rough_roll_dir = rolling_root / f"rough_sabr_logU0_seed{ROLLING_SEED}"
rolling_table_dir = TABLE_DIR / "rolling_windows"
rolling_figure_dir = FIGURE_DIR / "rolling_windows"
rolling_table_dir.mkdir(parents=True, exist_ok=True)
rolling_figure_dir.mkdir(parents=True, exist_ok=True)

normal_roll = pd.read_csv(normal_roll_dir / "filtered_state_path.csv").sort_values("t_index")
rough_roll = pd.read_csv(rough_roll_dir / "filtered_state_path.csv").sort_values("t_index")
forward_roll = (
    pd.read_csv(normal_roll_dir / "selected_observations.csv", usecols=["t_index", "F_obs"])
    .groupby("t_index", as_index=False)["F_obs"]
    .median()
)

window_start_t = int(min(normal_roll["t_index"].min(), rough_roll["t_index"].min()))
for frame in (normal_roll, rough_roll, forward_roll):
    frame["relative_t"] = frame["t_index"] - window_start_t

surface_roll_path = OUTPUT_ROOT / "rolling_windows" / "_comparison" / "surface_roll_event_study_summary.csv"
surface_rolls = pd.read_csv(surface_roll_path)
surface_rolls = surface_rolls[
    surface_rolls["roll_t_index"].between(normal_roll["t_index"].min(), normal_roll["t_index"].max())
].copy()
surface_rolls["relative_t"] = surface_rolls["roll_t_index"] - window_start_t

paired_roll_states = (
    normal_roll[
        ["t_index", "relative_t", "F", "atm_logvol_proxy", "rho", "nu", "fit_price_rmse"]
    ]
    .rename(
        columns={
            "F": "F_normal",
            "atm_logvol_proxy": "atm_logvol_proxy_normal",
            "rho": "rho_normal",
            "nu": "nu_normal",
            "fit_price_rmse": "fit_price_rmse_normal",
        }
    )
    .merge(
        rough_roll[
            [
                "t_index",
                "F",
                "atm_logvol_proxy",
                "rho",
                "nu",
                "memory_norm",
                "fit_price_rmse",
            ]
        ].rename(
            columns={
                "F": "F_rough",
                "atm_logvol_proxy": "atm_logvol_proxy_rough",
                "rho": "rho_rough",
                "nu": "nu_rough",
                "fit_price_rmse": "fit_price_rmse_rough",
            }
        ),
        on="t_index",
        how="inner",
    )
    .merge(forward_roll[["t_index", "F_obs"]], on="t_index", how="left")
)
paired_roll_states["F_diff_rough_minus_normal"] = paired_roll_states["F_rough"] - paired_roll_states["F_normal"]
paired_roll_states["F_error_normal"] = paired_roll_states["F_normal"] - paired_roll_states["F_obs"]
paired_roll_states["F_error_rough"] = paired_roll_states["F_rough"] - paired_roll_states["F_obs"]
paired_roll_states.to_csv(
    rolling_table_dir / f"{ROLLING_WINDOW_LABEL}_seed{ROLLING_SEED}_paired_filtered_states.csv",
    index=False,
)

display(paired_roll_states.head())


In [ ]:
normal_color = "#1c5982"
rough_color = "#d65225"
obs_color = "0.25"
diff_color = "#6b4c9a"
roll_color = "#b33a3a"

fig, axes = plt.subplots(6, 1, figsize=(12.5, 12.2), sharex=True)
x = paired_roll_states["relative_t"]

axes[0].plot(x, paired_roll_states["F_obs"], color=obs_color, linewidth=1.15, label="Observed forward")
axes[0].plot(x, paired_roll_states["F_normal"], color=normal_color, linewidth=1.0, label="Markovian")
axes[0].plot(x, paired_roll_states["F_rough"], color=rough_color, linewidth=1.0, label="Rough-SABR")
axes[0].set_ylabel("Forward", fontsize=15)
axes[0].legend(loc="upper right", frameon=False, fontsize=11)

axes[1].axhline(0.0, color="0.30", linestyle="--", linewidth=0.9)
axes[1].plot(x, paired_roll_states["F_diff_rough_minus_normal"], color=diff_color, linewidth=1.05)
axes[1].set_ylabel(r"$F^R-F^S$", fontsize=15)

axes[2].plot(x, paired_roll_states["atm_logvol_proxy_normal"], color=normal_color, linewidth=1.0, label="Markovian")
axes[2].plot(x, paired_roll_states["atm_logvol_proxy_rough"], color=rough_color, linewidth=1.0, label="Rough-SABR")
axes[2].set_ylabel("ATM log-vol", fontsize=15)
axes[2].legend(loc="upper right", frameon=False, fontsize=11)

axes[3].plot(x, paired_roll_states["rho_normal"], color=normal_color, linewidth=1.0, label="Markovian")
axes[3].plot(x, paired_roll_states["rho_rough"], color=rough_color, linewidth=1.0, label="Rough-SABR")
axes[3].set_ylabel(r"$\rho$", fontsize=15)
axes[3].legend(loc="upper right", frameon=False, fontsize=11)

axes[4].plot(x, paired_roll_states["nu_normal"], color=normal_color, linewidth=1.0, label="Markovian")
axes[4].plot(x, paired_roll_states["nu_rough"], color=rough_color, linewidth=1.0, label="Rough-SABR")
axes[4].set_ylabel(r"$\nu$", fontsize=15)
axes[4].legend(loc="upper right", frameon=False, fontsize=11)

axes[5].plot(x, paired_roll_states["memory_norm"], color=rough_color, linewidth=1.05, label="Rough memory norm")
axes[5].set_ylabel("Memory", fontsize=15)
axes[5].set_xlabel(f"Relative timestamp within {ROLLING_WINDOW_LABEL}", fontsize=17)
axes[5].legend(loc="upper right", frameon=False, fontsize=11)

for ax in axes:
    for _, roll in surface_rolls.iterrows():
        ax.axvline(roll["relative_t"], color=roll_color, linestyle="--", linewidth=1.0, alpha=0.50)
    ax.grid(alpha=0.20)
    ax.tick_params(axis="both", which="major", direction="in", length=8, width=1.2, labelsize=13)
    ax.tick_params(axis="both", which="minor", direction="in", length=4, width=1.0)
    ax.minorticks_on()

fig.tight_layout()
path = rolling_figure_dir / f"{ROLLING_WINDOW_LABEL}_seed{ROLLING_SEED}_filtered_state_diagnostic.png"
fig.savefig(path, bbox_inches="tight", dpi=220)
plt.show()
print("Saved:", project_relative(path))


## Notes

The figures are descriptive diagnostics for the filtered latent states.
